# Fold 2 submission — `fold2_seed42` (val 0.91048)

The strongest checkpoint on [Ronochieng/road-barbados-htr](https://huggingface.co/Ronochieng/road-barbados-htr) is **`checkpoints/fold2_seed42`**, uploaded with the commit message `fold2_seed42: val 0.91048`.

That 0.91048 is the training-time score: greedy decode (`num_beams=1`), a single image view, on at most 320 lines of fold 2. It is the number used to pick the checkpoint. It is not a public leaderboard score.

This notebook does not train. It:

1. Downloads that adapter (and `preds/test_fold2_seed42.csv` if a previous run already decoded the test set).
2. If those test predictions are already on the hub and cover every test id, turns them into a submission and stops.
3. Otherwise loads `Qwen/Qwen3-VL-8B-Instruct` plus the LoRA adapter and decodes `Test.csv` with the same line normalisation and prompt the adapter was trained on.
4. Writes `submission_fold2_seed42.csv`.

Run it on a GPU with **at least 40 GB** (L40S, A100, H100). Put `Test.csv` and `SampleSubmission.csv` next to the notebook. `images.zip` is downloaded on first run if the crops are not already under `data/`. Read `HF_TOKEN` from the environment or from `.env`. Do not paste the token into a cell.

`FAST_DECODE = False` uses 5 beams and the four test-time views from the main pipeline (about an hour on an L40S). Set it to `True` for one greedy view — the same protocol that produced 0.91048, and several times faster.


In [ ]:
import importlib.metadata as importlib_metadata
import importlib.util
import subprocess
import sys

# Qwen3-VL's processor changes visual-token layout across transformers releases.
# 4.57.1 is the pin the adapter was trained against.
PINS = {
    "transformers": "transformers==4.57.1",
    "peft": "peft>=0.17.0",
    "accelerate": "accelerate>=1.10.0",
    "cv2": "opencv-python-headless>=4.9",
    "PIL": "pillow>=10.2",
    "huggingface_hub": "huggingface_hub>=0.34.0",
    "pandas": "pandas>=2.1",
    "tqdm": "tqdm",
    "safetensors": "safetensors",
}


def _missing() -> list[str]:
    need = []
    for mod, spec in PINS.items():
        if importlib.util.find_spec(mod) is None:
            need.append(spec)
    try:
        if importlib_metadata.version("transformers") != "4.57.1":
            need.append("transformers==4.57.1")
    except importlib_metadata.PackageNotFoundError:
        need.append("transformers==4.57.1")
    # Keep order, drop duplicates.
    seen = set()
    out = []
    for spec in need:
        if spec not in seen:
            seen.add(spec)
            out.append(spec)
    return out


missing = _missing()
if missing:
    print("pip install", *missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("[env] inference packages already present (transformers 4.57.1)")


In [ ]:
import math
import os
import zipfile
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

# The hub folder behind "fold2_seed42: val 0.91048".
TAG = "fold2_seed42"
REPO_ID = os.environ.get("HF_REPO_ID", "Ronochieng/road-barbados-htr")
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

# False: 5 beams x 4 views, the submission decode from the main pipeline (~1 h).
# True:  1 beam x 1 view, the protocol that produced the 0.91048 validation number.
FAST_DECODE = False

LINE_HEIGHT = 160
MAX_LINE_WIDTH = 3200
MIN_LINE_WIDTH = 160
MIN_LINE_HEIGHT = 64
MAX_UPSCALE = 2.5
PATCH_GRID = 32
MAX_NEW_TOKENS = 64
LENGTH_PENALTY = 1.0
INFER_BATCH = 8 if not FAST_DECODE else 16
NUM_BEAMS = 1 if FAST_DECODE else 5
N_BEST = 1 if FAST_DECODE else 5
SCALES = (1.0,) if FAST_DECODE else (1.0, 1.22)
PHOTOMETRIC = not FAST_DECODE

WORK = Path(".").resolve()
OUT = WORK / "artifacts_fold2"
OUT.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUT / "cache" / f"h{LINE_HEIGHT}_w{MAX_LINE_WIDTH}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def load_dotenv(path: Path = WORK / ".env") -> None:
    """Existing environment variables win. The token never goes into this notebook."""
    if not path.exists():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key, val = key.strip(), val.strip().strip('"').strip("'")
        if key and val and not os.environ.get(key):
            os.environ[key] = val


load_dotenv()
TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not TOKEN:
    raise SystemExit("Export HF_TOKEN or put it in .env before running this notebook.")
print(f"[cfg] repo={REPO_ID}  tag={TAG}  fast={FAST_DECODE}  beams={NUM_BEAMS}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"[cfg] gpu={props.name}  vram={props.total_memory / 1e9:.1f} GB")
else:
    print("[cfg] no CUDA device visible — decode will refuse to start; a hub prediction file can still be formatted")


## 1. Pull the adapter

Only `checkpoints/fold2_seed42` is required. `preds/test_fold2_seed42.csv` is fetched too, so a test decode that already finished on the hub is reused instead of run again. `history/val_history_fold2_seed42.csv` is the table that contains the 0.91048 figure.


In [ ]:
from huggingface_hub import snapshot_download

HUB_DIR = OUT / "hub"
snapshot_download(
    repo_id=REPO_ID,
    repo_type="model",
    allow_patterns=[
        f"checkpoints/{TAG}/**",
        f"preds/test_{TAG}.csv",
        f"history/val_history_{TAG}.csv",
    ],
    local_dir=str(HUB_DIR),
    token=TOKEN,
)

hist_path = HUB_DIR / "history" / f"val_history_{TAG}.csv"
if hist_path.exists():
    hist = pd.read_csv(hist_path)
    score_cols = [c for c in hist.columns if c.endswith("_score")]
    if score_cols:
        best_row = hist.loc[hist[score_cols].max(axis=1).idxmax()]
        print(f"[hub] val history best row:\n{best_row.to_string()}")
    else:
        print(f"[hub] val history columns: {list(hist.columns)}")
else:
    print("[hub] no val history file — continuing with the adapter anyway")

ckpt = HUB_DIR / "checkpoints" / TAG
adapter_files = list(ckpt.rglob("adapter_config.json")) if ckpt.exists() else []
full_files = list(ckpt.rglob("*.safetensors")) if ckpt.exists() else []
if adapter_files:
    ADAPTER_DIR = adapter_files[0].parent
    print(f"[hub] LoRA adapter <- {ADAPTER_DIR}")
elif full_files:
    ADAPTER_DIR = full_files[0].parent
    print(f"[hub] full weights <- {ADAPTER_DIR}")
else:
    raise SystemExit(f"No weights under {ckpt}. The repo should contain checkpoints/{TAG}/.")

HUB_PRED = HUB_DIR / "preds" / f"test_{TAG}.csv"
print(f"[hub] cached test predictions: {HUB_PRED.exists()}")


## 2. Test images

The same crop index the training notebook uses. Training labels are not required to write a submission.


In [ ]:
DATA = WORK / "data"
DATA.mkdir(parents=True, exist_ok=True)
IMAGES_URL = "https://storage.googleapis.com/road-handwriting/images.zip"


def download(url: str, dest: Path) -> None:
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[data] {dest.name} already present ({dest.stat().st_size / 1e6:.0f} MB)")
        return
    import urllib.request
    print(f"[data] downloading {url}")
    tmp = dest.with_suffix(dest.suffix + ".part")
    with urllib.request.urlopen(url) as resp, open(tmp, "wb") as fh:
        while True:
            buf = resp.read(1 << 20)
            if not buf:
                break
            fh.write(buf)
    tmp.rename(dest)


def ensure_images() -> Path:
    img_dir = DATA / "images"
    if not (img_dir.exists() and any(img_dir.rglob("*.jpg"))):
        download(IMAGES_URL, DATA / "images.zip")
        print("[data] extracting ...")
        with zipfile.ZipFile(DATA / "images.zip") as zf:
            zf.extractall(DATA)
    for root in (img_dir, DATA):
        if root.exists() and any(root.rglob("*.jpg")):
            return next(root.rglob("*.jpg")).parent
    raise FileNotFoundError("No .jpg files found after extraction")


IMAGE_DIR = ensure_images()
IMAGE_INDEX = {
    p.stem: p for p in IMAGE_DIR.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
}
test_df = pd.read_csv(WORK / "Test.csv")
sample_sub = pd.read_csv(WORK / "SampleSubmission.csv")
test_df["ID"] = test_df["ID"].astype(str).str.strip()
sample_sub["ID"] = sample_sub["ID"].astype(str).str.strip()
missing = [i for i in test_df.ID if i not in IMAGE_INDEX]
print(f"[data] {IMAGE_DIR}  indexed {len(IMAGE_INDEX)}  test rows {len(test_df)}  missing {len(missing)}")
assert not missing, f"Test images missing: {missing[:5]}"


## 3. Line normalisation and prompt

Copied from the training notebook. Height 160, width cap 3200, 32-pixel grid, background flattening, deskew, horizontal ink crop, CLAHE. Changing any of these changes the picture the adapter learned to read.


In [ ]:
import unicodedata
import re

UNICODE_FOLD = {
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u2032": "'",
    "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u2033": '"',
    "\u2010": "-", "\u2011": "-", "\u2012": "-", "\u2013": "-",
    "\u2014": "-", "\u2015": "-", "\u2212": "-",
    "\u00a0": " ", "\u2007": " ", "\u202f": " ", "\u2009": " ", "\u200b": "",
    "\u2026": "...", "\ufeff": "",
}
_FOLD_RE = re.compile("|".join(map(re.escape, UNICODE_FOLD)))

OCR_PROMPT = (
    "This is a cropped line from a 17th-19th century handwritten Barbadian legal record "
    "(deed, will, inventory or census page).\n"
    "Transcribe the single complete line of handwriting that runs across the middle of the image. "
    "Partial lines clipped at the top and bottom edges are context only - do not transcribe them.\n"
    "Reproduce the original early-modern spelling, capitalisation and punctuation exactly as "
    "written; do not modernise or correct it. Keep the caret '^' that marks a superscript "
    "(for example y^e, 15^th, w^tsoever).\n"
    "Reply with the transcription of that line and nothing else."
)


def normalize_text(s) -> str:
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = unicodedata.normalize("NFC", str(s))
    s = _FOLD_RE.sub(lambda m: UNICODE_FOLD[m.group()], s)
    s = s.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    return " ".join(s.split()).strip()


def read_gray(path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        img = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img


def flatten_background(gray: np.ndarray) -> np.ndarray:
    k = max(11, (min(gray.shape[:2]) // 2) | 1)
    k = min(k, 51) | 1
    bg = cv2.medianBlur(gray, k)
    out = cv2.divide(gray, np.maximum(bg, 1), scale=255)
    return np.clip(out, 0, 255).astype(np.uint8)


def ink_mask(gray: np.ndarray) -> np.ndarray:
    g = cv2.GaussianBlur(gray, (0, 0), 1.0)
    m = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                              cv2.THRESH_BINARY_INV, 41, 12)
    return cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))


def estimate_skew(gray: np.ndarray, max_deg: float = 3.0, step: float = 0.5) -> float:
    h, w = gray.shape[:2]
    scale = 64.0 / max(h, 1)
    small = cv2.resize(gray, (max(int(w * scale), 32), 64), interpolation=cv2.INTER_AREA)
    mask = (ink_mask(small) > 0).astype(np.float32)
    if mask.sum() < 20:
        return 0.0
    best_angle, best_score = 0.0, -1.0
    centre = (mask.shape[1] / 2, mask.shape[0] / 2)
    for angle in np.arange(-max_deg, max_deg + 1e-6, step):
        if abs(angle) < 1e-6:
            rot = mask
        else:
            M = cv2.getRotationMatrix2D(centre, float(angle), 1.0)
            rot = cv2.warpAffine(mask, M, (mask.shape[1], mask.shape[0]),
                                 flags=cv2.INTER_NEAREST, borderValue=0)
        score = float(np.var(rot.sum(axis=1)))
        if score > best_score:
            best_score, best_angle = score, float(angle)
    return best_angle


def rotate_keep(gray: np.ndarray, angle: float) -> np.ndarray:
    if abs(angle) < 1e-3:
        return gray
    h, w = gray.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REPLICATE)


def horizontal_ink_crop(gray: np.ndarray, pad_frac: float = 0.02) -> np.ndarray:
    m = ink_mask(gray)
    col = m.sum(axis=0)
    thr = max(col.max() * 0.02, 1.0)
    on = np.nonzero(col > thr)[0]
    if on.size < 2:
        return gray
    pad = int(gray.shape[0] * pad_frac) + 2
    a = max(int(on[0]) - pad, 0)
    b = min(int(on[-1]) + pad, gray.shape[1])
    if b - a < gray.shape[0]:
        return gray
    return gray[:, a:b]


def snap(x: float, grid: int, lo: int, hi: int) -> int:
    v = int(round(x / grid)) * grid
    return int(min(max(v, max(lo, grid)), hi))


def target_size(w: int, h: int, scale: float = 1.0) -> tuple[int, int]:
    s = (LINE_HEIGHT * scale) / max(h, 1)
    s = min(s, MAX_UPSCALE)
    tw, th = w * s, h * s
    if tw > MAX_LINE_WIDTH:
        k = MAX_LINE_WIDTH / tw
        tw, th = tw * k, th * k
    th = snap(th, PATCH_GRID, MIN_LINE_HEIGHT, LINE_HEIGHT * 2)
    tw = snap(tw, PATCH_GRID, MIN_LINE_WIDTH, MAX_LINE_WIDTH)
    return int(tw), int(th)


def normalize_line(gray: np.ndarray, scale: float = 1.0) -> np.ndarray:
    gray = flatten_background(gray)
    gray = rotate_keep(gray, estimate_skew(gray))
    gray = horizontal_ink_crop(gray)
    gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    h, w = gray.shape[:2]
    tw, th = target_size(w, h, scale)
    interp = cv2.INTER_AREA if tw < w else cv2.INTER_CUBIC
    return cv2.resize(gray, (tw, th), interpolation=interp)


def load_line(image_id: str, scale: float = 1.0) -> np.ndarray:
    if abs(scale - 1.0) < 1e-6:
        cached = CACHE_DIR / f"{image_id}.png"
        if cached.exists():
            img = cv2.imread(str(cached), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                return img
    path = IMAGE_INDEX.get(str(image_id).strip())
    if path is None:
        return np.full((LINE_HEIGHT, MIN_LINE_WIDTH), 235, np.uint8)
    out = normalize_line(read_gray(path), scale)
    if abs(scale - 1.0) < 1e-6:
        cv2.imwrite(str(CACHE_DIR / f"{image_id}.png"), out)
    return out


def to_pil(gray: np.ndarray) -> Image.Image:
    return Image.fromarray(cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB))


def photometric_variant(img: np.ndarray, mode: str) -> np.ndarray:
    if mode == "identity":
        return img
    if mode == "dark":
        f = np.clip(255.0 * np.power(img / 255.0, 1.35), 0, 255).astype(np.uint8)
        return cv2.erode(f, np.ones((2, 2), np.uint8))
    if mode == "light":
        f = np.clip(255.0 * np.power(img / 255.0, 0.78), 0, 255).astype(np.uint8)
        return cv2.dilate(f, np.ones((2, 2), np.uint8))
    raise ValueError(mode)


print("[image] normalisation functions ready")


## 4. Decode

Each view is written to `artifacts_fold2/preds/` as it finishes, so a restarted kernel continues from the views already on disk. The hypothesis with the highest beam score is the submission string.


In [ ]:
PRED_DIR = OUT / "preds"
PRED_DIR.mkdir(parents=True, exist_ok=True)
TEST_IDS = test_df.ID.tolist()


def load_nbest(path: Path) -> dict:
    df = pd.read_csv(path)
    df["text"] = df["text"].fillna("")
    out = defaultdict(list)
    for row in df.itertuples(index=False):
        out[str(row.ID)].append({
            "text": normalize_text(row.text),
            "score": float(row.score),
            "view": str(getattr(row, "view", "1:identity")),
            "rank": int(getattr(row, "rank", 0)),
        })
    return dict(out)


def save_nbest(nbest: dict, path: Path) -> None:
    rows = []
    for image_id, cands in nbest.items():
        for c in cands:
            rows.append({"ID": image_id, **c})
    pd.DataFrame(rows).to_csv(path, index=False)


def top1(nbest: dict) -> dict:
    out = {}
    for image_id, cands in nbest.items():
        if not cands:
            out[image_id] = ""
            continue
        best = max(cands, key=lambda c: (c["score"], -c["rank"]))
        out[image_id] = best["text"]
    return out


def covers(nbest: dict, ids: list[str]) -> bool:
    return set(ids) <= set(nbest)


USE_HUB_PREDS = HUB_PRED.exists() and covers(load_nbest(HUB_PRED), TEST_IDS)
print(f"[decode] reuse hub test file: {USE_HUB_PREDS}")


def views():
    chosen = [(s, "identity") for s in SCALES]
    if PHOTOMETRIC:
        chosen.append((SCALES[0], "dark"))
        chosen.append((SCALES[0], "light"))
    return chosen


def view_path(scale: float, mode: str) -> Path:
    return PRED_DIR / f"test_{TAG}_x{scale:g}_{mode}.csv"


def build_model():
    if not torch.cuda.is_available():
        raise SystemExit(
            "Qwen3-VL-8B in bf16 needs a GPU with at least 40 GB. "
            "This machine has no CUDA device.")
    from transformers import AutoModelForImageTextToText, AutoProcessor
    from peft import PeftModel

    attn = "sdpa"
    try:
        import flash_attn  # noqa: F401
        if torch.cuda.get_device_properties(0).major >= 8:
            attn = "flash_attention_2"
    except Exception:
        pass

    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    ip = getattr(processor, "image_processor", None)
    if ip is not None:
        max_px = LINE_HEIGHT * 2 * MAX_LINE_WIDTH
        min_px = MIN_LINE_HEIGHT * MIN_LINE_WIDTH
        for attr, val in (("min_pixels", min_px), ("max_pixels", max_px)):
            if hasattr(ip, attr):
                setattr(ip, attr, val)
        if isinstance(getattr(ip, "size", None), dict):
            ip.size = {**ip.size, "shortest_edge": min_px, "longest_edge": max_px}
    processor.tokenizer.padding_side = "left"
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    is_adapter = (ADAPTER_DIR / "adapter_config.json").exists()
    source = MODEL_ID if is_adapter else str(ADAPTER_DIR)
    kwargs = dict(attn_implementation=attn, device_map={"": 0}, trust_remote_code=True)
    try:
        model = AutoModelForImageTextToText.from_pretrained(source, dtype=torch.bfloat16, **kwargs)
    except TypeError:
        model = AutoModelForImageTextToText.from_pretrained(source, torch_dtype=torch.bfloat16, **kwargs)
    if is_adapter:
        model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
        print(f"[model] LoRA <- {ADAPTER_DIR}  attn={attn}")
    else:
        print(f"[model] full weights <- {ADAPTER_DIR}  attn={attn}")
    model.eval()
    model.config.use_cache = True
    gc_ = model.generation_config
    gc_.pad_token_id = processor.tokenizer.pad_token_id
    eos = processor.tokenizer.eos_token_id
    im_end = processor.tokenizer.convert_tokens_to_ids("<|im_end|>")
    ids = [i for i in {eos, im_end} if isinstance(i, int) and i >= 0]
    gc_.eos_token_id = ids if len(ids) > 1 else (ids[0] if ids else eos)
    prompt = processor.apply_chat_template(
        [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": OCR_PROMPT}]}],
        tokenize=False, add_generation_prompt=True,
    )
    return model, processor, prompt


def clean_text(text: str) -> str:
    text = text.split("<|im_end|>")[0]
    for tag in ("assistant\n", "assistant:", "<|endoftext|>"):
        if text.startswith(tag):
            text = text[len(tag):]
    return normalize_text(text)


@torch.inference_mode()
def generate_batch(model, processor, prompt, images, num_beams, n_best):
    tokenizer = processor.tokenizer
    side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    try:
        inputs = processor(text=[prompt] * len(images), images=images,
                           return_tensors="pt", padding=True)
    finally:
        tokenizer.padding_side = side
    inputs = {k: (v.to(model.device) if hasattr(v, "to") else v) for k, v in inputs.items()}
    in_len = inputs["input_ids"].shape[1]
    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=num_beams,
        output_scores=True, return_dict_in_generate=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    if num_beams > 1:
        gen_kwargs.update(num_return_sequences=min(n_best, num_beams),
                          length_penalty=LENGTH_PENALTY, early_stopping=True)
    out = model.generate(**inputs, **gen_kwargs)
    seqs = out.sequences[:, in_len:]
    texts = processor.batch_decode(seqs, skip_special_tokens=True)
    if num_beams > 1 and getattr(out, "sequences_scores", None) is not None:
        scores = out.sequences_scores.float().cpu().tolist()
    else:
        scores = [0.0] * len(texts)
    k = min(n_best, num_beams) if num_beams > 1 else 1
    grouped = []
    for b in range(len(images)):
        grouped.append([(clean_text(texts[b * k + j]), float(scores[b * k + j])) for j in range(k)])
    del out
    return grouped


def decode_view(model, processor, prompt, scale: float, mode: str) -> dict:
    path = view_path(scale, mode)
    if path.exists():
        cached = load_nbest(path)
        if covers(cached, TEST_IDS):
            print(f"[decode] reuse {path.name}")
            return cached
    ids = list(TEST_IDS)
    widths = []
    for i in ids:
        p = CACHE_DIR / f"{i}.png"
        if p.exists():
            with Image.open(p) as im:
                widths.append(im.size[0])
        else:
            widths.append(10 ** 6)
    order = np.argsort(-np.asarray(widths))
    results = {}
    bs = max(1, INFER_BATCH // max(1, NUM_BEAMS // 2))
    pos = 0
    pbar = tqdm(total=len(ids), desc=f"x{scale:g},{mode}", unit="img")
    while pos < len(order):
        chunk = [ids[j] for j in order[pos:pos + bs]]
        try:
            images = []
            for i in chunk:
                img = load_line(i, scale=scale)
                if mode != "identity":
                    img = photometric_variant(img, mode)
                images.append(to_pil(img))
            grouped = generate_batch(model, processor, prompt, images, NUM_BEAMS, N_BEST)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs == 1:
                print(f"!! OOM on {chunk[0]} — empty string")
                results[chunk[0]] = [("", -50.0)]
                pos += 1
                pbar.update(1)
                continue
            bs = max(1, bs // 2)
            print(f"!! CUDA OOM — batch size {bs}")
            continue
        for image_id, cand in zip(chunk, grouped):
            results[image_id] = cand
        pos += len(chunk)
        pbar.update(len(chunk))
    pbar.close()
    nbest = {}
    view = f"{scale:g}:{mode}"
    for image_id in ids:
        cand = results.get(image_id, [("", -50.0)])
        nbest[image_id] = [
            {"text": text, "score": score, "view": view, "rank": rank}
            for rank, (text, score) in enumerate(cand)
        ]
    save_nbest(nbest, path)
    print(f"[decode] wrote {path.name}")
    return nbest


if USE_HUB_PREDS:
    NBEST = load_nbest(HUB_PRED)
    print(f"[decode] {len(NBEST)} ids from {HUB_PRED.name}")
else:
    todo = [i for i in TEST_IDS if not (CACHE_DIR / f"{i}.png").exists()]
    if todo:
        print(f"[cache] normalising {len(todo)} test lines")
        for i in tqdm(todo, desc="cache", unit="img"):
            load_line(i)
    pending = [vm for vm in views() if not (view_path(*vm).exists() and covers(load_nbest(view_path(*vm)), TEST_IDS))]
    model = processor = prompt = None
    if pending:
        model, processor, prompt = build_model()
    pooled = defaultdict(list)
    for scale, mode in views():
        part = decode_view(model, processor, prompt, scale, mode)
        for image_id, cands in part.items():
            pooled[image_id].extend(cands)
    NBEST = dict(pooled)
    save_nbest(NBEST, PRED_DIR / f"test_{TAG}.csv")
    if model is not None:
        del model
        torch.cuda.empty_cache()
print(f"[decode] hypotheses per line: {np.mean([len(NBEST[i]) for i in TEST_IDS]):.1f}")


## 5. Submission

Rows follow `SampleSubmission.csv`. An empty transcription is replaced with `the` so the row is not scored as a total miss; the count of those rows is printed.


In [ ]:
pred = top1(NBEST)
submission = sample_sub[["ID"]].copy()
submission["Target"] = submission["ID"].map(lambda i: pred.get(i, ""))
submission["Target"] = submission["Target"].fillna("").map(normalize_text)

problems = []
if len(submission) != len(sample_sub):
    problems.append(f"row count {len(submission)} != sample {len(sample_sub)}")
if set(submission.ID) != set(sample_sub.ID):
    problems.append("ID set does not match SampleSubmission.csv")
n_empty = int((submission.Target.str.strip() == "").sum())
if n_empty:
    problems.append(f"{n_empty} empty predictions filled with 'the'")
    submission.loc[submission.Target.str.strip() == "", "Target"] = "the"

print("submission checks:", "OK" if not problems else "")
for p in problems:
    print("  !!", p)
lengths = submission.Target.str.len()
print(f"length: median {lengths.median():.0f}  p10 {lengths.quantile(0.1):.0f}  p90 {lengths.quantile(0.9):.0f}")

out_path = WORK / f"submission_{TAG}.csv"
submission.to_csv(out_path, index=False)
print(f"\nwrote {out_path} ({len(submission)} rows)")
print(submission.head(8).to_string(index=False))
